In [1]:
from os import listdir as ls
import pandas as pd
import json
import yaml as yml
import pickle
import pycountry
import pycountry_convert as pc
import random
import arviz as az
from matplotlib import pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
from IPython.display import display, Markdown, Latex

from emu_renewal.constants import DATA_PATH, FULL_RUN, TIMEOUTS, RERUNS, WANING_COMPARISON, WANING_TIMEOUTS, WANING_RERUNS, \
    MOB_LOCATION_SOURCE_MAP, MOB_LOCATION_NAME_MAP, CONT_CMAP, BASE_PATH, OUTPUTS_PATH, ANALYSIS_NAMES
from emu_renewal.inputs import get_world_shp
from emu_renewal.outputs import load_targets, get_country_procs, get_param_vals_by_analysis, get_idatas_for_mob_type, \
    get_ratios_from_disps, get_median_ratios, get_quantmedian_df, convert_quant_df_for_display
from emu_renewal.plotting import plot_multianalysis_fit, add_cont_to_world_geodf, plot_continent_grouping, plot_prior_multipost, \
    compare_proc_mob, compare_proc_weighted_gmob, plot_proc_comparison, plot_kde_comparison, plot_mob_exp_violins, \
    plot_waning_comparison_proc_disp, plot_waning_quant_comparison, plot_input_recovery
from emu_renewal.utils import get_countries_by_continent, get_analysis_paths, get_analysis_commits_df, split_list_into_segments, \
    get_country_short_name

set_matplotlib_formats("svg")

# Calibration results
## Purpose
This document presents the calibration to data
for each country, analysis and target indicator for fitting.

In [ ]:
display(Latex(r"\newpage"))

## Country grouping
For this document and the remainder of this supplemental file,
we grouped countries according to the following continent groupings.

In [ ]:
#| fig-align: center
world = get_world_shp()
add_cont_to_world_geodf(world)
plt.style.use("default")
plot_continent_grouping(world)

In [ ]:
display(Latex(r"\newpage"))

# Results by continent and country

In [ ]:
#| fig-align: center
plt.style.use("ggplot")
all_countries = json.load(open(DATA_PATH / "config/included.json", "r"))
analysis_paths = get_analysis_paths(RERUNS + FULL_RUN + TIMEOUTS, all_countries)
# analysis_paths = get_analysis_paths(WANING_RERUNS + WANING_COMPARISON + WANING_TIMEOUTS, all_countries)
countries_by_cont = get_countries_by_continent(all_countries)
countries_by_cont = {"AF": ["DZA", "AGO"]} # *** TEMPORARY PATCH TO REDUCE RUN TIME FOR NOTEBOOK ***
for cont, countries in countries_by_cont.items():
    display(Latex(r"\newpage"))
    display(Markdown(f"## {pc.convert_continent_code_to_continent_name(cont)}"))
    for c, country in enumerate(countries):
        country_name = pycountry.countries.lookup(country).name
        if c:
            display(Latex(r"\newpage"))
        display(Markdown(f"### {country_name}"))
        analyses = analysis_paths[country]
        
        no_mob_path = analyses["no_mob"]
    
        spaghs = {a: pd.read_hdf(p / "spaghetti.h5") for a, p in analyses.items()}
        targs = load_targets(no_mob_path)
        display(plot_multianalysis_fit(targs, spaghs))

# Purpose
This document presents comparisons of parameter prior distributions 
to their corresponding posterior distributions.

In [ ]:
display(Latex(r"\newpage"))

# Prior posterior comparisons by continent and country

In [ ]:
#| fig-align: center
for co, (cont, countries) in enumerate(countries_by_cont.items()):
    if co:
        display(Latex(r"\newpage"))
    display(Markdown(f"\n## {pc.convert_continent_code_to_continent_name(cont)}"))
    for c, country in enumerate(countries):
        country_name = pycountry.countries.lookup(country).name
        if c:
            display(Latex(r"\newpage"))
        display(Markdown(f"### {country_name}"))
        analyses = analysis_paths[country]

        no_mob_path = analyses["no_mob"]
        targs = load_targets(no_mob_path)
            
        priors = pickle.load(open(no_mob_path / "priors.pkl", "rb"))
        idatas = {a: az.from_netcdf(p / "idata_filtered.nc") for a, p in analyses.items()}
        var_names = ["start"] + [v[5:] for v in targs.keys() if v.startswith("prop_")]
        display(plot_prior_multipost(var_names, priors, idatas, 4))

In [ ]:
notes = {
    "g_mob": "Mobility values presented as Google data divided by 100, plus one.",
    "fb_visited_mob": "Mobility values presented as one plus Facebook data.",
    "fb_singletile_mob": "Mobility values presented as one minus Facebook data.",
}

# Purpose
This document presents the residual transmission scaling (with uncertainty) implemented without scaling for mobility for each country. It is intended to provide a raw comparison between the residual non-mechanistic transmission that needed to be applied for each analysis in the absence of mobility scaling and the various mobility data fields available from both Google and Facebook. We also show comparisons against the composite Google mobility metric after weighting, with associated credible intervals.

In [ ]:
display(Markdown("\\newpage # Individual mobility metric comparisons (Google and Facebook)\n\n"))
for m, mob_type in enumerate(MOB_LOCATION_SOURCE_MAP):
    mob_source = MOB_LOCATION_SOURCE_MAP[mob_type]
    mob_name = MOB_LOCATION_NAME_MAP[mob_type]
    if m:
        display(Latex(r"\newpage"))
    section_title = f"## {mob_name} mobility metric comparison\n\n"
    display(Markdown(section_title))
    display(Markdown(notes[mob_source]))
    mob_avail_countries = [c for c in all_countries if mob_source in analysis_paths[c]]
    for cont in ["AF"]:
    # for cont in CONT_CMAP:
        cont_countries = [c for c in countries_by_cont[cont] if c in mob_avail_countries]
        display(Markdown(f"### {pc.convert_continent_code_to_continent_name(cont)}"))
        for countries in split_list_into_segments(cont_countries, 12):
            display(compare_proc_mob(analysis_paths, countries, 4, mob_type))

mob_avail_countries = [c for c in all_countries if "g_mob" in analysis_paths[c]]
display(Markdown("# Weighted scaling process versus residual comparison\n\n"))
for cont in ["AF"]:
# for cont in CONT_CMAP:
    display(Markdown(f"### {pc.convert_continent_code_to_continent_name(cont)}"))
    
    cont_countries = [c for c in countries_by_cont[cont] if c in mob_avail_countries]
    for countries in split_list_into_segments(cont_countries, 16):
        display(compare_proc_weighted_gmob(analysis_paths, countries, 200, 4))

# Purpose
This document presents comparisons of residual transmission scaling
across each mobility analysis approach for each country,
along with the 95% credible interval associated with each.
We would expect that approaches to analysis that
if applying mobility captured some part of the variation 
in transmission that would otherwise need to be 
included through residual variation,
then that mobility approach would be associated with
less dramatic changes in the residual transmission scaling.

# Results by continent

In [ ]:
procs = get_country_procs(analysis_paths, all_countries)
for cont in countries_by_cont:
    cont_name = pc.convert_continent_code_to_continent_name(cont)
    display(Markdown(f"## {cont_name}"))
    cont_countries = countries_by_cont[cont]
    for countries in split_list_into_segments(cont_countries, 9):
        display(plot_proc_comparison(procs, countries, analysis_paths))

# Purpose
This document presents results based on the posterior distribution 
of the residual transmission scaling dispersion parameter.
This quantity governs the distribution in the change in residual scaling for transmission
from one value in the process series to the subsequent update.
As such, smaller values imply that smaller updates could still
result in good calibrations.
Lower values for this parameter can therefore be interpreted
as less dramatic changes needing to be applied through
the non-mechanistic component of the model.
As such, we interpreted mobility analysis approaches 
for which this posterior distribution of this parameter
was lower as being a more plausible representation of reality.
This is intended as a more formal quantification of the 
differences in variation in the non-mechanistic transmission scaling
presented in the previous document.

# Dispersion parameter distributions by analysis and country

In [ ]:
disp_posts = {c: get_param_vals_by_analysis("dispersion_proc", analysis_paths[c]) for c in all_countries}
param = "dispersion_proc"
note = (
    "Note that for the countries of Oceania, a different baseline analysis "
    "was used for to compare Google and Facebook analyses against. "
    "This was done because the conclusion of the Facebook data meant "
    "that the analysis period differed from the Google analysis."
)
param_name = yml.safe_load(open(DATA_PATH / "evidence/priors.yml", "r"))["other"]["dispersion_proc"]["short_name"]
axis_adjustments = {"URY": [-0.05, 0.35]}
for c, (cont, cont_countries) in enumerate(countries_by_cont.items()):
    if c:
        display(Latex(r"\newpage"))
    cont_name = pc.convert_continent_code_to_continent_name(cont)
    display(Markdown(f"## {cont_name}"))
    title = f"Posterior distribution for {param_name} parameter, {cont_name}"
    if cont == "OC":
        display(Markdown(note))
    for countries in split_list_into_segments(cont_countries, 16):
        param_posts = {iso3: disp_posts[iso3] for iso3 in countries}
        display(plot_kde_comparison(param_posts, axis_adjustments)) 

# Purpose
This document presents estimates of 
the mobility exponent parameter posteriors by country and mobility analysis type.
Values from zero to two were considered plausible 
(the prior for this parameter was uniform over domain $[0, 2]$). 
A value of zero represents no effect, 
a value of one represents a linear scaling in transmission with mobility changes,
and a value of two represents a quadratic scaling in transmission with mobility changes 
(which could be epidemiologically considered as an effect on both infector and infectee).
For all violin plots, the vertical axis represents the mobility exponent parameter distribution,
while the darkness of the coloured shading represents the median of the relative
values of the dispersion parameter posterior under the analysis with mobility included
compared against the baseline simulation.

# Mobility exponent parameter posterior distributions by country

In [ ]:
disp_posts = {c: get_param_vals_by_analysis("dispersion_proc", analysis_paths[c]) for c in all_countries}
ratio_dists = get_ratios_from_disps(disp_posts)

In [ ]:
analyses = {k: v for k, v in ANALYSIS_NAMES.items() if "no_mob" not in k}

for mob_source, mob_name in analyses.items():

    # Median ratios
    ratio_vals = get_median_ratios(ratio_dists, mob_source)
    ratios = {get_country_short_name(k): v for k, v in ratio_vals.items()}
    
    # Mobility exponent distributions
    idatas, _ = get_idatas_for_mob_type(analysis_paths, all_countries, mob_source)
    mob_exp_df = pd.DataFrame({c: idatas[c].posterior["mob_exp"].to_series() for c in idatas})

    # Plot
    display(Markdown(f"## {mob_name}"))
    display(plot_mob_exp_violins(mob_source, mob_exp_df, ratios))

In [ ]:
all_countries = json.load(open(DATA_PATH / f"config/included.json", "r"))
analysis_paths = get_analysis_paths(RERUNS + FULL_RUN + TIMEOUTS, all_countries)
waning_paths = get_analysis_paths(WANING_RERUNS + WANING_COMPARISON + WANING_TIMEOUTS, all_countries)
n_samples = 12
sample_countries = random.sample(list(analysis_paths), n_samples)
sample_analyses = [[iso3, random.sample(list(analysis_paths[iso3]), 1)[0]] for iso3 in sample_countries]

# Purpose
This document presents comparisons of key model parameters with and without the inclusion of waning immunity.

\newpage

# Example dispersion parameter posterior distributions
The following plots show the posterior distribution of the dispersion to the variable process
with and without the incorporation of waning immunity into the model.

In [ ]:
n_analyses = sum(len(p) for p in analysis_paths.values())
txt = f"These are presented for a random sample of country-analysis combinations sampled from the {n_analyses} total analyses, "
txt += f"with {n_samples} random analyses were selected for display."
Markdown(txt)

These are presented for a random sample of country-analysis combinations sampled from the 463 total analyses, with 12 random analyses were selected for display.

In [ ]:
plot_waning_comparison_proc_disp(waning_paths, analysis_paths, n_samples, sample_analyses)

{{< pagebreak >}}

# Commits used for analyses
For reproducbility, the following table gives the (short) commit SHA for each analysis.

In [ ]:
Markdown(get_analysis_commits_df(analysis_paths).to_markdown())